# Exercise 3: Neural networks in PyTorch

In this exercise you’ll implement small neural-network building blocks from scratch and use them to train a simple classifier.

You’ll cover:
- **Basic layers**: Linear, Embedding, Dropout
- **Normalization**: LayerNorm and RMSNorm
- **MLPs + residual**: composing layers into deeper networks
- **Classification**: generating a learnable dataset, implementing cross-entropy from logits, and writing a minimal training loop

As before: fill in all `TODO`s without changing function names or signatures.
Use small sanity checks and compare to PyTorch reference implementations when useful.

In [16]:
from __future__ import annotations

import torch
from torch import nn
import torch.nn.functional as F   # only used to sanity-check our layers against torch

## Basic layers

In this section you’ll implement a few core layers that appear everywhere:

### `Linear`
A fully-connected layer that follows nn.Linear conventions:  
`y = x @ Wᵀ + b`

Important details:
- Parameters should be registered as `nn.Parameter`
- Store weight as (out_features, in_features) like nn.Linear.
- The forward pass should support leading batch dimensions: `x` can be shape `(..., in_features)`

### `Embedding`
An embedding table maps integer ids to vectors:
- input: token ids `idx` of shape `(...,)`
- output: vectors of shape `(..., embedding_dim)`

This is essentially a learnable lookup table.

### `Dropout`
Dropout randomly zeroes activations during training to reduce overfitting.
Implementation details:
- Only active in `model.train()` mode
- In training: drop with probability `p` and scale the kept values by `1/(1-p)` so the expected value stays the same
- In eval: return the input unchanged

## Instructions
- Do not use PyTorch reference modules for the parts you implement (e.g. don’t call nn.Linear inside your Linear).
- You may use standard tensor ops that you learned before (matmul, sum, mean, rsqrt, indexing, etc.).
- Use a parameter initialization method of your choice. We recommend something like Xavier-uniform.


In [17]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        # weight is stored as (out, in), so forward is x @ W^T
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        # register_parameter(None) keeps `self.bias` defined even when there is no bias
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter("bias", None)

    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}"

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., in_features)
        return: (..., out_features)
        """
        y = x @ self.weight.T          # matmul broadcasts over all leading dims of x
        if self.bias is not None:
            y = y + self.bias          # (out,) broadcasts over (..., out)
        return y

# ---- y = x @ W^T + b, for any number of leading batch dims ----
torch.manual_seed(0)
lin = Linear(4, 3)
x = torch.randn(2, 5, 4)                         # (batch, seq, in_features)
print("x", tuple(x.shape), "-> y", tuple(lin(x).shape))
print("matches F.linear:", torch.allclose(lin(x), F.linear(x, lin.weight, lin.bias)))
print("registered parameters:", [(n, tuple(p.shape)) for n, p in lin.named_parameters()])
print("bias=False ->", [n for n, _ in Linear(4, 3, bias=False).named_parameters()])

x (2, 5, 4) -> y (2, 5, 3)
matches F.linear: True
registered parameters: [('weight', (3, 4)), ('bias', (3,))]
bias=False -> ['weight']


In [18]:
class Embedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        # one learnable row per id, initialised from N(0, 1)
        self.weight = nn.Parameter(torch.randn(num_embeddings, embedding_dim))

    def extra_repr(self) -> str:
        return f"{self.num_embeddings}, {self.embedding_dim}"

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """
        idx: (...,) int64
        return: (..., embedding_dim)
        """
        return self.weight[idx]         # advanced indexing: (...,) ids -> (..., embedding_dim)

# ---- a lookup table: row i of the weight IS the vector for id i ----
torch.manual_seed(0)
emb = Embedding(10, 3)
idx = torch.tensor([[1, 2, 2], [7, 0, 1]])        # (2, 3) token ids
out = emb(idx)
print("idx", tuple(idx.shape), "-> out", tuple(out.shape))
print("out[0, 1] equals weight[2]:", torch.equal(out[0, 1], emb.weight[2]))
print("matches F.embedding:", torch.equal(out, F.embedding(idx, emb.weight)))
# gradient only reaches the rows that were looked up (id 2 twice -> its grad is summed)
out.sum().backward()
print("rows with nonzero grad:", emb.weight.grad.abs().sum(1).nonzero().flatten().tolist(),
      " grad of row 2:", emb.weight.grad[2].tolist())

idx (2, 3) -> out (2, 3, 3)
out[0, 1] equals weight[2]: True
matches F.embedding: True
rows with nonzero grad: [0, 1, 2, 7]  grad of row 2: [2.0, 2.0, 2.0]


In [19]:
class Dropout(nn.Module):
    def __init__(self, p: float):
        super().__init__()
        assert 0.0 <= p <= 1.0, "p must be a probability"
        self.p = p

    def extra_repr(self) -> str:
        return f"p={self.p}"

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        In train mode: drop with prob p and scale by 1/(1-p).
        In eval mode: return x unchanged.
        """
        if not self.training or self.p == 0.0:
            return x                                   # eval mode (or p=0): identity
        if self.p == 1.0:
            return torch.zeros_like(x)                 # everything dropped; avoid dividing by 0
        keep = torch.rand_like(x) >= self.p            # True with probability 1-p
        return x * keep / (1.0 - self.p)               # inverted dropout keeps E[out] == x

# ---- train: zero ~p of the entries and scale the rest; eval: identity ----
torch.manual_seed(0)
drop = Dropout(p=0.25)
x = torch.ones(10_000)
drop.train()
y = drop(x)
print(f"train  fraction zeroed {(y == 0).float().mean().item():.3f} (p=0.25)"
      f"   kept values {y[y != 0].unique().tolist()} (=1/(1-p))   mean {y.mean().item():.3f} (~1)")
drop.eval()
print("eval   output is the input, untouched:", torch.equal(drop(x), x))

train  fraction zeroed 0.245 (p=0.25)   kept values [1.3333333730697632] (=1/(1-p))   mean 1.007 (~1)
eval   output is the input, untouched: True


## Normalization

Normalization layers help stabilize training by controlling activation statistics.

### LayerNorm
LayerNorm normalizes each example across its **feature dimension** (the last dimension):

- compute mean and variance over the last dimension
- normalize: `(x - mean) / sqrt(var + eps)`
- apply learnable per-feature scale and shift (`weight`, `bias`)

**In this exercise, assume `elementwise_affine=True` (always include `weight` and `bias`).**  
`weight` and `bias` each have shape `(D,)`.

LayerNorm is widely used in transformers because it does not depend on batch statistics.

### RMSNorm
RMSNorm is similar to LayerNorm but normalizes using only the root-mean-square:
- `x / sqrt(mean(x^2) + eps)` over the last dimension
- usually includes a learnable scale (`weight`)
- no mean subtraction

RMSNorm is popular in modern LLMs because it's faster.


In [20]:
class LayerNorm(nn.Module):
    def __init__(
        self, normalized_shape: int, eps: float = 1e-5, elementwise_affine: bool = True
    ):
        super().__init__()
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if elementwise_affine:
            # start as the identity transform: scale 1, shift 0
            self.weight = nn.Parameter(torch.ones(normalized_shape))
            self.bias = nn.Parameter(torch.zeros(normalized_shape))
        else:
            self.register_parameter("weight", None)
            self.register_parameter("bias", None)

    def extra_repr(self) -> str:
        return f"{self.normalized_shape}, eps={self.eps}"

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Normalize over the last dimension.
        x: (..., D)
        """
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)   # population (biased) variance
        x_hat = (x - mean) * torch.rsqrt(var + self.eps)
        if self.elementwise_affine:
            x_hat = x_hat * self.weight + self.bias
        return x_hat

# ---- every row (each token / each example) gets mean 0 and std 1 over its features ----
torch.manual_seed(0)
ln = LayerNorm(6)
x = 5 + 3 * torch.randn(2, 4, 6)                 # far from zero-mean / unit-std
y = ln(x)
print("per-row mean ~0:", round(y.mean(-1).abs().max().item(), 6),
      "  per-row std ~1:", round(y.std(-1, unbiased=False).mean().item(), 4))
with torch.no_grad():
    ln.weight.uniform_(0.5, 2.0); ln.bias.uniform_(-1, 1)   # non-trivial affine to make the check meaningful
print("matches F.layer_norm:", torch.allclose(ln(x), F.layer_norm(x, (6,), ln.weight, ln.bias, 1e-5), atol=1e-6))
print("no batch statistics: row 0 alone gives the same result:", torch.allclose(ln(x[:1]), ln(x)[:1]))

per-row mean ~0: 0.0   per-row std ~1: 1.0
matches F.layer_norm: True
no batch statistics: row 0 alone gives the same result: True


In [21]:
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-8):
        super().__init__()
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))

    def extra_repr(self) -> str:
        return f"{self.normalized_shape}, eps={self.eps}"

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        RMSNorm: x / sqrt(mean(x^2) + eps) * weight
        over the last dimension.
        """
        rms_inv = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x * rms_inv * self.weight

# ---- only rescale by the root-mean-square; the mean is NOT removed ----
torch.manual_seed(0)
rms = RMSNorm(6)
x = 5 + 3 * torch.randn(2, 6)
y = rms(x)
print("per-row RMS after ~1:", [round(v, 4) for v in y.pow(2).mean(-1).sqrt().tolist()])
print("per-row mean after  :", [round(v, 3) for v in y.mean(-1).tolist()], " <- not centred, unlike LayerNorm")
with torch.no_grad():
    rms.weight.uniform_(0.5, 2.0)
print("matches F.rms_norm:", torch.allclose(rms(x), F.rms_norm(x, (6,), rms.weight, eps=1e-8), atol=1e-6))

per-row RMS after ~1: [1.0, 1.0]
per-row mean after  : [0.691, 0.944]  <- not centred, unlike LayerNorm
matches F.rms_norm: True


## MLPs and residual networks

Now you’ll build larger networks by composing layers.

### MLP
An MLP is a stack of `depth` Linear layers with non-linear activations (use GELU) in between.
In this exercise you’ll support:
- configurable depth
- a hidden dimension
- optional LayerNorm between layers (a common stabilization trick)

A key skill is building networks using `nn.ModuleList` / `nn.Sequential` while keeping shapes consistent.

### Transformer-style FeedForward (FFN)
A transformer block contains a position-wise feedforward network:
- `D -> 4D -> D` (by default)
- activation is typically **GELU**

This is essentially an MLP applied independently at each token position.

### Residual wrapper
Residual connections are the simplest form of “skip connection”:
- output is `x + fn(x)`

They improve gradient flow and allow training deeper networks more reliably.

In [22]:
class MLP(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        out_dim: int,
        depth: int,
        use_layernorm: bool = False,
    ):
        super().__init__()
        assert depth >= 1, "an MLP needs at least one Linear layer"
        # depth Linear layers: in -> hidden -> ... -> hidden -> out   (depth=1 means in -> out)
        dims = [in_dim] + [hidden_dim] * (depth - 1) + [out_dim]
        layers: list[nn.Module] = []
        for i, (d_in, d_out) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(Linear(d_in, d_out))
            is_last = i == depth - 1
            if not is_last:                   # no norm / activation after the output layer
                if use_layernorm:
                    layers.append(LayerNorm(d_out))
                layers.append(nn.GELU())
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

# ---- depth counts Linear layers; GELU (and optional LayerNorm) sit only BETWEEN them ----
mlp = MLP(in_dim=8, hidden_dim=32, out_dim=3, depth=3, use_layernorm=True)
print(mlp.net)
print("x (5, 8) -> y", tuple(mlp(torch.randn(5, 8)).shape))
print("depth=1 is a single Linear:", [type(m).__name__ for m in MLP(8, 32, 3, depth=1).net])

Sequential(
  (0): Linear(in_features=8, out_features=32, bias=True)
  (1): LayerNorm(32, eps=1e-05)
  (2): GELU(approximate='none')
  (3): Linear(in_features=32, out_features=32, bias=True)
  (4): LayerNorm(32, eps=1e-05)
  (5): GELU(approximate='none')
  (6): Linear(in_features=32, out_features=3, bias=True)
)
x (5, 8) -> y (5, 3)
depth=1 is a single Linear: ['Linear']


In [23]:
class FeedForward(nn.Module):
    """
    Transformer-style FFN: D -> 4D -> D (default)
    """

    def __init__(self, d_model: int, d_ff: int | None = None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = Linear(d_model, d_ff)   # expand
        self.act = nn.GELU()
        self.fc2 = Linear(d_ff, d_model)   # project back

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.act(self.fc1(x)))

# ---- D -> 4D -> D, applied to every token position independently ----
torch.manual_seed(0)
ffn = FeedForward(d_model=16)
x = torch.randn(2, 5, 16)                        # (batch, tokens, d_model)
print("x", tuple(x.shape), "-> y", tuple(ffn(x).shape), "  hidden width", ffn.fc1.out_features)
print("position-wise: token 3 alone gives the same output:", torch.allclose(ffn(x[:, 3]), ffn(x)[:, 3], atol=1e-6))

x (2, 5, 16) -> y (2, 5, 16)   hidden width 64
position-wise: token 3 alone gives the same output: True


In [24]:
class Residual(nn.Module):
    def __init__(self, fn: nn.Module):
        super().__init__()
        self.fn = fn

    def forward(self, x: torch.Tensor, *args, **kwargs) -> torch.Tensor:
        return x + self.fn(x, *args, **kwargs)

# ---- x + fn(x): even if fn contributes nothing, the input (and its gradient) passes straight through ----
torch.manual_seed(0)
block = Residual(FeedForward(16))
x = torch.randn(3, 16, requires_grad=True)
print("shape kept:", tuple(block(x).shape))
with torch.no_grad():
    block.fn.fc2.weight.zero_(); block.fn.fc2.bias.zero_()   # make fn output exactly 0
print("fn == 0  ->  output equals input:", torch.equal(block(x), x))
block(x).sum().backward()
print("and dL/dx is all ones (the identity path):", torch.equal(x.grad, torch.ones_like(x)))

shape kept: (3, 16)
fn == 0  ->  output equals input: True
and dL/dx is all ones (the identity path): True


## Classification problem

In this section you’ll put everything together in a minimal MNIST classification experiment.

You will:
1) download and load the MNIST dataset
2) implement cross-entropy from logits (stable, using log-softmax)
3) build a simple MLP-based classifier (flatten MNIST images first)
4) write a minimal training loop
5) report train loss curve and final accuracy

The goal here is not to reach state-of-the-art accuracy, but to understand the full pipeline:
data → model → logits → loss → gradients → parameter update.

### Model notes
- We want you to combine the MLP we implemented above with the classification head we define below into one model 

### MNIST notes
- MNIST images are `28×28` grayscale.
- After `ToTensor()`, each image has shape `(1, 28, 28)` and values in `[0, 1]`.
- For an MLP classifier, we flatten to a vector of length `784`.

## Deliverables
- Include a plot of your train loss curve in the video submission as well as a final accuracy. 
- **NOTE** Here we don't grade on model performance but we expect you to achieve at least 70% accuracy to confirm a correct model implementation.

In [25]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [26]:
transform = transforms.ToTensor()  # -> float32 in [0,1], shape (1, 28, 28)

train_ds = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)    # reshuffled every epoch
test_loader = DataLoader(test_ds, batch_size=1000, shuffle=False)   # order does not matter for evaluation

images, labels = next(iter(train_loader))
print(f"{len(train_ds)} train / {len(test_ds)} test images")
print("one batch:", tuple(images.shape), images.dtype, f"values in [{images.min().item():.0f}, {images.max().item():.0f}]",
      "  labels", tuple(labels.shape), labels.dtype)

60000 train / 10000 test images
one batch: (128, 1, 28, 28) torch.float32 values in [0, 1]   labels (128,) torch.int64


In [27]:
def cross_entropy_from_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    """
    Compute mean cross-entropy loss from logits.

    logits: (B, C)
    targets: (B,) int64

    Requirements:
    - Use log-softmax for stability (do not use torch.nn.CrossEntropyLoss, we check this in the autograder).
    """
    # log_softmax = logits - logsumexp(logits): stable even for huge logits (no exp overflow)
    log_probs = torch.log_softmax(logits, dim=-1)                  # (B, C)
    # pick the log-probability of the correct class in every row
    correct = log_probs.gather(dim=1, index=targets.unsqueeze(1))  # (B, 1)
    return -correct.mean()

# ---- loss = -log p(correct class), averaged over the batch ----
logits = torch.tensor([[2.0, 0.5, -1.0],
                       [0.1, 0.2,  3.0]])
targets = torch.tensor([0, 2])
naive = -torch.log(torch.softmax(logits, dim=-1)[torch.arange(2), targets]).mean()
print(f"ours {cross_entropy_from_logits(logits, targets).item():.6f}   naive softmax->log {naive.item():.6f}")
print(f"uniform logits over 10 classes -> {cross_entropy_from_logits(torch.zeros(4, 10), torch.zeros(4, dtype=torch.long)).item():.4f}"
      f" = log(10) = {torch.log(torch.tensor(10.0)).item():.4f}   (the loss at initialisation)")
# why log-softmax: the naive version overflows to inf / nan for large logits
big = logits * 1000
naive_big = -torch.log(torch.exp(big) / torch.exp(big).sum(-1, keepdim=True))[torch.arange(2), targets].mean()
print(f"logits x1000: ours {abs(cross_entropy_from_logits(big, targets).item()):.4f}   naive exp/sum {naive_big.item()}")

ours 0.175456   naive softmax->log 0.175456
uniform logits over 10 classes -> 2.3026 = log(10) = 2.3026   (the loss at initialisation)
logits x1000: ours 0.0000   naive exp/sum nan


In [28]:
class ClassificationHead(nn.Module):
    def __init__(self, d_in: int, num_classes: int):
        super().__init__()
        self.proj = Linear(d_in, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., d_in)
        return: (..., num_classes) logits
        """
        return self.proj(x)       # raw scores (logits); softmax lives inside the loss

In [29]:
@torch.no_grad()                       # evaluation: no graph, less memory, faster
def evaluate_accuracy(model: nn.Module, loader: DataLoader) -> float:
    """Fraction of correctly classified examples in `loader`."""
    model.eval()                       # dropout off, norms in inference mode
    device = next(model.parameters()).device
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=-1)  # most likely class per example
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / total


@torch.no_grad()
def evaluate_loss(model: nn.Module, loader: DataLoader) -> float:
    """Mean cross-entropy over every example in `loader`."""
    model.eval()
    device = next(model.parameters()).device
    total_loss = total = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        total_loss += cross_entropy_from_logits(model(x), y).item() * y.numel()   # undo the per-batch mean
        total += y.numel()
    return total_loss / total


def accuracy(loader):
    # evaluates the global `model` defined in the experiment cell below
    return evaluate_accuracy(model, loader)

In [30]:
def train_loader_iter(loader: DataLoader, device: torch.device):
    """Yield batches already moved to `device`."""
    for x, y in loader:
        yield x.to(device), y.to(device)


def train_classifier(
    model: nn.Module,
    train_data_loader: DataLoader,
    test_data_loader: DataLoader,
    lr: float,
    epochs: int,
    seed: int = 0,
) -> list[float]:
    """
    Minimal training loop for MNIST classification.

    Steps:
    - define optimizer
    - for each epoch:
        - sample minibatches
        - forward -> cross-entropy -> backward -> optimizer step
      - compute test accuracy at the end of each epoch
    - return list of training losses (one per update step)

    Requirements:
    - call model.train() during training and model.eval() during evaluation
    - do not use torch.nn.CrossEntropyLoss (use your cross_entropy_from_logits)
    """
    torch.manual_seed(seed)                         # reproducible shuffling / dropout
    device = next(model.parameters()).device
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses: list[float] = []
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader_iter(train_data_loader, device):
            logits = model(x)                       # forward
            loss = cross_entropy_from_logits(logits, y)
            optimizer.zero_grad()
            loss.backward()                         # backward
            optimizer.step()                        # update
            losses.append(loss.item())
        test_acc = evaluate_accuracy(model, test_data_loader)   # switches to eval() internally
        test_loss = evaluate_loss(model, test_data_loader)
        n = len(train_data_loader)
        print(f"epoch {epoch + 1}/{epochs}   mean train loss {sum(losses[-n:]) / n:.4f}"
              f"   test loss {test_loss:.4f}   test acc {test_acc:.4f}")
    model.eval()
    return losses

In [31]:
# ---- put it together: flatten (1, 28, 28) -> 784, MLP body, linear classification head ----
torch.manual_seed(0)
model = nn.Sequential(
    nn.Flatten(),                                                     # (B, 1, 28, 28) -> (B, 784)
    MLP(in_dim=28 * 28, hidden_dim=256, out_dim=256, depth=2, use_layernorm=True),
    nn.GELU(),
    ClassificationHead(d_in=256, num_classes=10),
)
print("parameters:", sum(p.numel() for p in model.parameters()))
print(f"test accuracy before training: {accuracy(test_loader):.4f}  (~0.1 = random guessing)")
print()

losses = train_classifier(model, train_loader, test_loader, lr=1e-3, epochs=3, seed=0)

print()
print(f"{len(losses)} update steps   first loss {losses[0]:.3f} (~log 10 = 2.303)   last loss {losses[-1]:.3f}")
print(f"final test accuracy: {accuracy(test_loader):.4f}")

parameters: 269834
test accuracy before training: 0.1177  (~0.1 = random guessing)

epoch 1/3   mean train loss 0.1976   test loss 0.0960   test acc 0.9699
epoch 2/3   mean train loss 0.0755   test loss 0.0769   test acc 0.9760
epoch 3/3   mean train loss 0.0501   test loss 0.0776   test acc 0.9760

1407 update steps   first loss 2.384 (~log 10 = 2.303)   last loss 0.069
final test accuracy: 0.9760
